In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("dirty_cafe_sales.csv")

In [ ]:
data.info()

### Correcting the column data type

In [ ]:
data["Quantity"] = pd.to_numeric(data["Quantity"], errors='coerce')

data["Price Per Unit"] = pd.to_numeric(data["Price Per Unit"], errors='coerce')

data["Total Spent"] = pd.to_numeric(data["Total Spent"], errors='coerce')

data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], errors='coerce')

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data["Item"].unique()

#### Fixing "Price Per Unit" by "Total Spent" and "Quantity"

In [ ]:
missing_price = ( data["Price Per Unit"].isna() & data["Total Spent"].notna() & data["Quantity"].notna() & (data["Quantity"] != 0))

data.loc[missing_price, "Price Per Unit"] = (data.loc[missing_price, "Total Spent"] / data.loc[missing_price, "Quantity"])

#### Fixing "Price Per Unit" by Item name

In [ ]:
item_to_price = {
    "Coffee": 2.0,
    "Cake": 3.0,
    "Cookie": 1.0,
    "Salad": 5.0,
    "Smoothie": 4.0,
    "Sandwich": 4.0,
    "Juice": 3.0,
    "Tea": 1.5
}

missing_price = data["Price Per Unit"].isna()

data.loc[missing_price, "Price Per Unit"] = (data.loc[missing_price, "Item"].map(item_to_price))


#### Fixing "Item name" by "Price Per Unit"

In [ ]:
price_to_item_mode = (
    data
    .dropna(subset=["Item", "Price Per Unit"])
    .groupby("Price Per Unit")["Item"]
    .agg(lambda x: x.mode()[0])
)

missing_item = ((data["Item"].isna() | data["Item"].isin(["ERROR", "UNKNOWN"])) & data["Price Per Unit"].notna())

data.loc[missing_item, "Item"] = (data.loc[missing_item, "Price Per Unit"].map(price_to_item_mode))

#### Fixing "Quantity" by "Total Spent" and "Price Per Unit"

In [ ]:
missing_qty = (data["Quantity"].isna() & data["Total Spent"].notna() & data["Price Per Unit"].notna() & (data["Price Per Unit"] != 0))

data.loc[missing_qty, "Quantity"] = (data.loc[missing_qty, "Total Spent"] / data.loc[missing_qty, "Price Per Unit"])

#### Fixing "Total Spent" by "Price Per Unit" and "Quantity"

In [ ]:
missing_total = (data["Total Spent"].isna() & data["Quantity"].notna() & data["Price Per Unit"].notna())

data.loc[missing_total, "Total Spent"] = (data.loc[missing_total, "Quantity"] * data.loc[missing_total, "Price Per Unit"])

#### Find "Quantity" and "Price Per Unit" by "Item name" and "Total Spent"

In [ ]:
special_case = (data["Item"].notna() & data["Total Spent"].notna() & data["Price Per Unit"].isna() & data["Quantity"].isna())

data.loc[special_case, "Price Per Unit"] = (data.loc[special_case, "Item"].map(item_to_price))

data.loc[special_case, "Quantity"] = (data.loc[special_case, "Total Spent"] / data.loc[special_case, "Price Per Unit"])

### Drop the remaining rows where it is impossible to find the item name or determine its price.

In [ ]:
irreparable_rows = (data[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum(axis=1) >= 2)

data = data[~irreparable_rows]

data.isnull().sum()

### Fill in the "Payment Method" and "Location" fields that are empty or have incorrect values ​​with the most frequently used values.

In [ ]:
payment_method_mode = data["Payment Method"].mode()

data["Payment Method"] = data["Payment Method"].replace(["ERROR", "UNKNOWN"], payment_method_mode[0])

data["Payment Method"] = data["Payment Method"].fillna(payment_method_mode[0])

data.isnull().sum()

In [ ]:
location_mode = data["Location"].mode()

data["Location"] = data["Location"].replace(["ERROR", "UNKNOWN"], location_mode[0])

data["Location"] = data["Location"].fillna(location_mode[0])

data.isnull().sum()

### Delete incorrect dates

In [ ]:
invalid_mask = data["Transaction Date"].isna()
data[invalid_mask]

In [ ]:
data = data.dropna(subset=["Transaction Date"])
data.isnull().sum()

### Add a column for the seasons based on the "Transaction Date"

In [ ]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

data["Season"] = data["Transaction Date"].apply(get_season)

In [ ]:
data.head()

### Save the clean data in a new CSV file

In [ ]:
data.to_csv("Cleaned_Cafe_Sales.csv", index=False)

In [ ]:
data.info()

In [ ]:
data.isnull().sum()